In [7]:
import pandas as pd

from power import panel_mde
from tabulate import tabulate

In [8]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"
mde_out = "../../../reports/replication/mde.md"

In [9]:
panel = pd.concat([
    pd.read_parquet(train_panel_in),
    pd.read_parquet(bike_panel_in),
    pd.read_parquet(uber_panel_in),
    ], keys=['train','bike','uber'], names=['transit'])

In [10]:
mde_stats = panel_mde(panel, ['transit'], 'id', 'rides', 'UCMP == 1')

In [11]:
# Format for presentation.
mde_stats.index.set_names(['Transit'], inplace=True)
mde_stats['Mean Rides per Unit per Day (SD)'] = mde_stats['ar_mean'].apply("{:.1f}".format) + \
                                                mde_stats['ar_std'].apply(" ({:.1f})".format)
mde_stats['Minimum Detectable Change'] = "+" + mde_stats['mdc'].apply("{:.0%}".format)
mde_stats['Total Additional Rides'] = mde_stats['mds'].apply("{:.0f}".format)
mde_stats.drop(columns=['ar_mean','ar_std','mdc','mds',
                        'geo_mean','exp(geo_mean)', 'geo_std',
                        'se', 'exp(se)', 'mde', 'n'], inplace=True)
mde_stats

,Mean Rides per Unit per Day (SD),Minimum Detectable Change,Total Additional Rides
Transit,,,
bike,49.8 (22.3),+143%,9311
train,1383.0 (540.6),+167%,38237
uber,820.1 (751.3),+130%,53566


In [12]:

# Note: For future reference if you need multi-header columns:
# The best option is to tabualte with tablefmt=html, open the file and
# added a new <tr> in the table header, fixed the column name labels, 
# and used the <th colspan="2"> attribute to make the spanning columns.

with open(mde_out,"w") as f:
    tbl = tabulate(mde_stats.reset_index(), 
                   headers=mde_stats.reset_index().columns, 
                   showindex=False,
                   tablefmt='github')
    f.write(tbl)
    print(tbl)

!cp {mde_out} ../../../../eric-mc2-cv/static/uploads/

| Transit   | Mean Rides per Unit per Day (SD)   | Minimum Detectable Change   |   Total Additional Rides |
|-----------|------------------------------------|-----------------------------|--------------------------|
| bike      | 49.8 (22.3)                        | +143%                       |                     9311 |
| train     | 1383.0 (540.6)                     | +167%                       |                    38237 |
| uber      | 820.1 (751.3)                      | +130%                       |                    53566 |
